In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance

In [ ]:
# laod dataset
df = pd.read_csv(r"/content/Telco-Customer-Churn.csv")
df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [ ]:
# convert TotalCharges into float values
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)
print(df['TotalCharges'].dtypes)

float64


In [ ]:
X = df.drop(["customerID", "Churn"], axis=1)
y = df['Churn']

In [ ]:
cat_cols = [col for col in X.columns if X[col].dtype == 'object']
num_cols = [col for col in X.columns if X[col].dtype != 'object']
print("numerical cols= ", num_cols, "\n")
print("categorical cols= ", cat_cols)

numerical cols=  ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'] 

categorical cols=  ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [ ]:
# remove "SeniorCitizen" from numerical cols
num_cols.remove("SeniorCitizen")
print(num_cols)

['tenure', 'MonthlyCharges', 'TotalCharges']


In [ ]:
# split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=42)
print(X_train.shape, X_test.shape)

(5493, 19) (1550, 19)


In [ ]:
# encode the y
y_train = y_train.map({'Yes': 1, 'No': 0})
y_train.tail(10)

,Churn
5334,1
466,0
6265,0
5734,0
3092,1
3772,1
5191,0
5226,0
5390,1
860,0


In [ ]:
y_test = y_test.map({'Yes': 1, 'No': 0})
y_test.tail(10)

,Churn
5936,0
4366,0
3589,0
5162,0
5229,1
5167,0
3288,0
6698,0
2132,0
6243,0


In [ ]:
# create pipeline but pass the "SeniorCitizen" from encoding
num_pipeline = Pipeline([
    ('std_scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('one_hot_encoder', OneHotEncoder())
])

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder = "passthrough")

In [ ]:
xgb = XGBClassifier(random_state=42, n_jobs=-1, verbose=2)
xgb_pipeline = Pipeline([
    ('full_pipeline', full_pipeline),
    ('xgb', xgb)
]).fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:43:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# print performance metrics
acc = accuracy_score(y_train, xgb_pipeline.predict(X_train))
precision = precision_score(y_train, xgb_pipeline.predict(X_train))
recall = recall_score(y_train, xgb_pipeline.predict(X_train))
f1 = f1_score(y_train, xgb_pipeline.predict(X_train))
print(f"Accuracy: {acc}\nPrecision: {precision}\nRecall: {recall}\nF1 Score: {f1}")

Accuracy: 0.9424722373930456
Precision: 0.9109985528219972
Recall: 0.8670798898071626
F1 Score: 0.8884968242766408


In [ ]:
r = permutation_importance(xgb_pipeline, X_test, y_test,
                           n_repeats=10,
                           random_state=42)

In [ ]:
%pip install eli5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 3.3 MB/s eta 0:00:00


In [ ]:
import eli5
transformed_feature_names = xgb_pipeline.named_steps['full_pipeline'].get_feature_names_out()
eli5.show_weights(xgb_pipeline.named_steps['xgb'], feature_names=transformed_feature_names.tolist())

Weight,Feature
0.4269,cat__Contract_Month-to-month
0.1749,cat__InternetService_Fiber optic
0.0343,cat__OnlineSecurity_No
0.0316,cat__Contract_Two year
0.0241,cat__PhoneService_No
0.0226,cat__StreamingMovies_Yes
0.0207,cat__TechSupport_No
0.0185,cat__Contract_One year
0.0163,num__tenure
0.0142,cat__OnlineBackup_No


In [ ]:
import pandas as pd

# Get feature importances from the XGBoost model
feature_importances = xgb_pipeline.named_steps['xgb'].feature_importances_

# Get the transformed feature names from the full_pipeline
transformed_feature_names = xgb_pipeline.named_steps['full_pipeline'].get_feature_names_out()

# Create a DataFrame to display feature importances
importance_df = pd.DataFrame({
    'Feature': transformed_feature_names,
    'Importance': feature_importances
})

# Sort by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

display(importance_df)

,Feature,Importance
0,cat__Contract_Month-to-month,0.426870
1,cat__InternetService_Fiber optic,0.174917
2,cat__OnlineSecurity_No,0.034330
3,cat__Contract_Two year,0.031609
4,cat__PhoneService_No,0.024113
5,cat__StreamingMovies_Yes,0.022608
6,cat__TechSupport_No,0.020701
7,cat__Contract_One year,0.018461
8,num__tenure,0.016337
9,cat__OnlineBackup_No,0.014211
